# CM4 Carrier Board — Power & Signal Integrity Analysis

Foundation board for an outdoor self-contained sensor system: battery + solar powered, cellular uplink, CM4 running basic computer vision (RGB camera on CSI) and thermal imaging (FLIR Lepton on SPI). The carrier exposes CM4 interfaces via headers — downstream boards connect the camera, thermal sensor, cellular modem (RS-485 to Modbus gateway or SPI), and solar charge controller (I2C BMS telemetry).

This notebook derives power rail voltage budget under load, polyfuse margin, I2C pull-up selection, and RS-485 termination/bias values for the carrier board.

## Assumptions
- 5V input from solar charge controller or battery regulator (no mains)
- Ambient temperature range -20°C to +50°C (outdoor enclosure, no polyfuse derating applied — worst case analysis at room temp)
- CM4 load modeled as constant-current sink (vision processing draws steady ~1.4A, peaks to 3A on boot)
- PCB trace resistance ~20 mOhm (2-layer, 1oz Cu, short runs)
- I2C bus capacitance ~100pF (BMS + thermal sensor + 60pF trace)
- RS-485 cable impedance 120 Ohm (shielded twisted pair to Modbus gateway)
- Parasitic inductance neglected (short traces, bulk caps present)

In [1]:
# Section 1: Power rail voltage budget (symbolic)
import sympy as sp

V_in, I_load = sp.symbols("V_in I_load", positive=True)
R_fuse, R_fet, R_trace = sp.symbols("R_fuse R_fet R_trace", positive=True)

# Reason: series path — screw terminal -> polyfuse -> MOSFET -> CM4
V_rail = V_in - I_load * (R_fuse + R_fet + R_trace)
print(f"V_rail = {V_rail}")
print(f"Total drop = {sp.simplify(V_in - V_rail)}")

V_rail = -I_load*(R_fet + R_fuse + R_trace) + V_in
Total drop = I_load*(R_fet + R_fuse + R_trace)


$$V_{rail} = V_{in} - I_{load} \cdot (R_{fuse} + R_{FET} + R_{trace}) \quad [\text{V}]$$

$$\Delta V_{drop} = I_{load} \cdot (R_{fuse} + R_{FET} + R_{trace}) \quad [\text{V}]$$

Where $V_{in}$ = supply voltage [V], $I_{load}$ = CM4 current draw [A], $R_{fuse}$, $R_{FET}$, $R_{trace}$ = series path resistances [$\Omega$].

In [2]:
# Numerical evaluation — power rail + polyfuse margin
import pint
from uncertainties import ufloat

ureg = pint.UnitRegistry()

v_in = 5.0 * ureg.V
i_typical = 1.4 * ureg.A
i_peak = 3.0 * ureg.A

r_fuse = ufloat(0.050, 0.010) * ureg.ohm
r_fet = ufloat(0.085, 0.015) * ureg.ohm
r_trace = ufloat(0.020, 0.005) * ureg.ohm
r_total = r_fuse + r_fet + r_trace

# Rail voltage at typical and peak load
v_rail_typical = v_in - i_typical * r_total
v_rail_peak = v_in - i_peak * r_total
v_cm4_min = 4.75 * ureg.V

print(f"R_total = {r_total.to(ureg.mohm)}")
print(f"V_rail (typical 1.4A) = {v_rail_typical:.3f}")
print(f"V_rail (peak 3.0A) = {v_rail_peak:.3f}")
print(f"CM4 minimum = {v_cm4_min}")

# Polyfuse margin
i_hold = 2.0 * ureg.A
print(
    f"\nI_typical / I_hold = {(i_typical / i_hold).to(ureg.dimensionless):.2f} (< 1 = safe)"
)
print(
    f"I_peak / I_trip = {(i_peak / (4.0 * ureg.A)).to(ureg.dimensionless):.2f} (< 1 = no trip)"
)

R_total = 155+/-19 milliohm
V_rail (typical 1.4A) = 4.783+/-0.026 volt
V_rail (peak 3.0A) = 4.535+/-0.056 volt
CM4 minimum = 4.75 volt

I_typical / I_hold = 0.70 dimensionless (< 1 = safe)
I_peak / I_trip = 0.75 dimensionless (< 1 = no trip)


## Section 2: Signal Integrity — I2C Pull-Up and RS-485 Termination

In [3]:
# I2C pull-up resistor selection — fast mode 400kHz
# Reason: R_max from rise time, R_min from sink current
# NXP UM10204 Rev. 7.0, Table 10

v_cc = 3.3  # V
v_ol = 0.4  # V
i_sink = 3e-3  # A, fast mode
c_bus = 100e-12  # F, estimated 4 devices + traces
t_rise_max = 300e-9  # s, fast mode spec

# R_min: must not exceed sink current capability
r_min = (v_cc - v_ol) / i_sink
print(f"R_pullup_min = {r_min:.0f} Ohm ({r_min / 1e3:.2f} kOhm)")

# R_max: rise time constraint (RC time constant)
# t_rise = 0.8473 * R * C (for RC charging to 70% of Vcc)
r_max = t_rise_max / (0.8473 * c_bus)
print(f"R_pullup_max = {r_max:.0f} Ohm ({r_max / 1e3:.1f} kOhm)")

# Select standard E24 value within range
r_selected = 2.2e3  # 2.2 kOhm
print(f"\nSelected: R_pullup = {r_selected / 1e3:.1f} kOhm")
print(
    f"In range: {r_min:.0f} < {r_selected:.0f} < {r_max:.0f} = {r_min < r_selected < r_max}"
)

# Verify rise time with selected value
t_rise_actual = 0.8473 * r_selected * c_bus
print(
    f"Actual rise time = {t_rise_actual * 1e9:.0f} ns (max {t_rise_max * 1e9:.0f} ns)"
)

R_pullup_min = 967 Ohm (0.97 kOhm)
R_pullup_max = 3541 Ohm (3.5 kOhm)

Selected: R_pullup = 2.2 kOhm
In range: 967 < 2200 < 3541 = True
Actual rise time = 186 ns (max 300 ns)


In [4]:
# RS-485 termination and failsafe bias
# Reason: termination = Z0 of cable, bias ensures defined idle state
# MAX3485E datasheet, receiver threshold = +/-200mV differential

r_term = 120  # Ohm, matches Z0 of twisted pair
v_cc_rs485 = 3.3  # V
r_bias = 560  # Ohm, pullup on A, pulldown on B

# Failsafe bias voltage at receiver input when bus is idle (no driver)
# Thevenin equivalent: V_diff = Vcc * R_term / (R_term + 2*R_bias) (approx)
# Reason: A pulled to Vcc via R_bias, B pulled to GND via R_bias, R_term across A-B
v_diff_idle = v_cc_rs485 * r_term / (r_term + 2 * r_bias)
print(f"R_term = {r_term} Ohm (= Z0)")
print(f"R_bias = {r_bias} Ohm (A->Vcc, B->GND)")
print(f"Idle bus V_diff = {v_diff_idle * 1e3:.0f} mV")
print("Receiver threshold = +/-200 mV")
print(f"Margin = {(v_diff_idle - 0.2) * 1e3:.0f} mV ({v_diff_idle > 0.2})")

R_term = 120 Ohm (= Z0)
R_bias = 560 Ohm (A->Vcc, B->GND)
Idle bus V_diff = 319 mV
Receiver threshold = +/-200 mV
Margin = 119 mV (True)


## Expected Values

| Parameter | Value | Tolerance | Source |
|-----------|-------|-----------|--------|
| V_rail at typical load (1.4A) | 4.78 V | +/-26 mV | Power budget calc |
| V_rail at peak load (3.0A) | 4.54 V | +/-56 mV | Power budget calc |
| I_typical / I_hold | 0.70 | — | Polyfuse margin (< 1 = safe) |
| I_peak / I_trip | 0.75 | — | Polyfuse margin (< 1 = no trip) |
| I2C R_pullup range | 0.97 - 3.5 kOhm | — | NXP UM10204 Table 10 |
| I2C R_pullup selected | 2.2 kOhm | E24 standard | Within calculated range |
| I2C rise time | 186 ns | — | Must be < 300 ns |
| RS-485 R_term | 120 Ohm | — | = Z0 of twisted pair |
| RS-485 idle V_diff | 319 mV | — | Must be > 200 mV threshold |

**Design note:** V_rail at 3A peak drops below CM4 minimum (4.75V). This is acceptable for brief boot surges (< 100ms) supported by bulk capacitance. Sustained 3A draw requires a lower-Rds MOSFET or direct power (no polyfuse in high-current path).